In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model 必须能被 n_heads 整除"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads          # 每个头分到的维度
        # 四组可学习投影：Q、K、V、以及输出投影 W_o
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, L, _ = x.shape                      # 批次, 序列长度, 维度
        # 投影后拆成多个头： [B,L,d_model] -> [B, n_heads, L, d_k]
        Q = self.W_q(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)

        # 昨天那四步公式，只是现在多了一个 head 维度
        scores  = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)   # [B,h,L,L]
        weights = F.softmax(scores, dim=-1)                        # 每行和为1
        out     = weights @ V                                      # [B,h,L,d_k]

        # 把多个头拼回来，再过一次输出投影
        out = out.transpose(1, 2).contiguous().view(B, L, self.d_model)
        return self.W_o(out)

# 快速验证
mha = MultiHeadAttention(d_model=128, n_heads=4)
x = torch.randn(2, 10, 128)     # 2个样本, 10个词, 每个词128维
print("输入:", x.shape, "→ 输出:", mha(x).shape)   # 应仍是 [2,10,128]

输入: torch.Size([2, 10, 128]) → 输出: torch.Size([2, 10, 128])


In [2]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()      # [max_len,1]
        # 不同维度用不同频率的正弦波
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)   # 偶数维用 sin
        pe[:, 1::2] = torch.cos(pos * div)   # 奇数维用 cos
        self.register_buffer('pe', pe)       # 注册为"常量"，不参与训练

    def forward(self, x):
        return x + self.pe[:x.size(1)]       # 直接相加：位置上叠一层"坐标"

pe = PositionalEncoding(16, max_len=5)
print("位置编码前5个位置:\n", pe.pe[:5].round(decimals=2))

位置编码前5个位置:
 tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000,
          0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8400,  0.5400,  0.3100,  0.9500,  0.1000,  1.0000,  0.0300,  1.0000,
          0.0100,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.9100, -0.4200,  0.5900,  0.8100,  0.2000,  0.9800,  0.0600,  1.0000,
          0.0200,  1.0000,  0.0100,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.1400, -0.9900,  0.8100,  0.5800,  0.3000,  0.9600,  0.0900,  1.0000,
          0.0300,  1.0000,  0.0100,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [-0.7600, -0.6500,  0.9500,  0.3000,  0.3900,  0.9200,  0.1300,  0.9900,
          0.0400,  1.0000,  0.0100,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000]])


In [3]:
class Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        # 前馈层：本质就是您熟悉的 MLP
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        x = self.ln1(x + self.attn(x))   # 注意力 + 残差 + 归一化
        x = self.ln2(x + self.ff(x))     # 前馈层 + 残差 + 归一化
        return x

block = Block(d_model=128, n_heads=4, d_ff=256)
x = torch.randn(2, 10, 128)
print("Block 输出:", block(x).shape)     # 仍是 [2,10,128]

Block 输出: torch.Size([2, 10, 128])


In [4]:
text = """the sun is bright. the sky is blue. the bird can fly.
the bird flies over the tree. the tree is tall and green.
the cat can run. the cat runs after the bird.
the bird can fly away. the cat can not fly.
the sun is warm. the tree is green. the bird is free.
"""

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s): return [stoi[c] for c in s]
data = torch.tensor(encode(text), dtype=torch.long)
print(f"字符表大小: {vocab_size}, 文本长度: {len(data)}")
print("字符表:", ''.join(chars))

字符表大小: 24, 文本长度: 256
字符表: 
 .abcdefghiklmnorstuvwy


In [5]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, d_ff=256,
                 n_layers=3, max_len=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)   # 字符 -> 向量
        self.pos   = PositionalEncoding(d_model, max_len)
        self.blocks = nn.Sequential(*[Block(d_model, n_heads, d_ff)
                                      for _ in range(n_layers)])
        self.ln    = nn.LayerNorm(d_model)
        self.head  = nn.Linear(d_model, vocab_size)      # 预测下一个字符

    def forward(self, x):
        x = self.pos(self.embed(x))     # 查表 + 加位置
        x = self.blocks(x)              # 堆叠多个 Block
        x = self.ln(x)
        return self.head(x)             # [B, L, vocab_size]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = MiniGPT(vocab_size).to(device)
print(f"参数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"运行设备: {device}")

参数量: 403,864
运行设备: cuda


In [6]:
block_size = 32
batch_size = 32

def get_batch():
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

x, y = get_batch()
print("输入:", x.shape, " 目标:", y.shape)
print("示例：输入第0行前12字 →", ''.join(itos[i] for i in x[0][:12].tolist()))
print("     对应目标前12字 →", ''.join(itos[i] for i in y[0][:12].tolist()))

输入: torch.Size([32, 32])  目标: torch.Size([32, 32])
示例：输入第0行前12字 → s blue. the 
     对应目标前12字 →  blue. the b


In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for step in range(1000):
    x, y = get_batch()
    logits = model(x)                                   # [B, L, vocab]
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    optimizer.zero_grad()
    loss.backward()          # 反向穿过整台 Transformer
    optimizer.step()
    if (step+1) % 200 == 0:
        print(f"step {step+1}: loss = {loss.item():.4f}")

step 200: loss = 0.4678
step 400: loss = 0.0605
step 600: loss = 0.0115
step 800: loss = 0.0052
step 1000: loss = 0.0022


In [8]:
@torch.no_grad()
def generate(start="the ", max_new=100):
    model.eval()
    idx = torch.tensor([encode(start)], dtype=torch.long, device=device)
    for _ in range(max_new):
        idx_cond = idx[:, -block_size:]
        logits = model(idx_cond)[:, -1, :]              # 只取最后一位
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)  # 按概率采样
        idx = torch.cat([idx, next_id], dim=1)
    return ''.join(itos[i] for i in idx[0].tolist())

print(generate("the ", 150))

the the t the. the the thcat the cat can run. t the cat runs after the bird.
the bird can fly away. the cat can not fly.
the isun is warm. the tree is gre
